# Topic Modeling und Themenanalyse

In diesem Notebook werden die häufigsten Themen innerhalb der Beschwerdetexte mithilfe semantischer Analyseverfahren identifiziert. Dafür werden die Verfahren Latent Dirichlet Allocation (LDA) und Latent Semantic Analysis (LSA) verwendet.

## Import der benötigten Bibliotheken

Für die Themenanalyse werden Verfahren aus scikit-learn sowie gespeicherte Vektormatrizen verwendet.

In [1]:
import pandas as pd
import pickle

from sklearn.decomposition import LatentDirichletAllocation
from sklearn.decomposition import TruncatedSVD

In [2]:
df = pd.read_csv("../processed_data/02_preprocessed_data.csv")

## Laden der Vektormatrizen

Die in Notebook 3 erzeugten Vektormatrizen werden geladen und für die Themenanalyse verwendet.

In [3]:
with open("../processed_data/03_bow_vectors.pkl", "rb") as f:
    X_bow = pickle.load(f)

with open("../processed_data/03_tfidf_vectors.pkl", "rb") as f:
    X_tfidf = pickle.load(f)

In [4]:
with open("../processed_data/03_bow_vectorizer.pkl", "rb") as f:
    bow_vectorizer = pickle.load(f)

with open("../processed_data/03_tfidf_vectorizer.pkl", "rb") as f:
    tfidf_vectorizer = pickle.load(f)

## Themenanalyse mit Latent Dirichlet Allocation (LDA)

LDA gruppiert Dokumente anhand häufig gemeinsam auftretender Begriffe in verschiedene Themenbereiche.

In [5]:
lda = LatentDirichletAllocation(
    n_components=5,
    random_state=42
)

In [6]:
lda.fit(X_bow)

,"n_components n_components: int, default=10Number of topics... versionchanged:: 0.19 ``n_topics`` was renamed to ``n_components``",5
,"doc_topic_prior doc_topic_prior: float, default=NonePrior of document topic distribution `theta`. If the value is None,defaults to `1 / n_components`.In [1]_, this is called `alpha`.",None
,"topic_word_prior topic_word_prior: float, default=NonePrior of topic word distribution `beta`. If the value is None, defaultsto `1 / n_components`.In [1]_, this is called `eta`.",None
,"learning_method learning_method: {'batch', 'online'}, default='batch'Method used to update `_component`. Only used in :meth:`fit` method.In general, if the data size is large, the online update will be muchfaster than the batch update.Valid options:- 'batch': Batch variational Bayes method. Use all training data in each EM update. Old `components_` will be overwritten in each iteration.- 'online': Online variational Bayes method. In each EM update, use mini-batch of training data to update the ``components_`` variable incrementally. The learning rate is controlled by the ``learning_decay`` and the ``learning_offset`` parameters... versionchanged:: 0.20 The default learning method is now ``""batch""``.",'batch'
,"learning_decay learning_decay: float, default=0.7It is a parameter that control learning rate in the online learningmethod. The value should be set between (0.5, 1.0] to guaranteeasymptotic convergence. When the value is 0.0 and batch_size is``n_samples``, the update method is same as batch learning. In theliterature, this is called kappa.",0.7
,"learning_offset learning_offset: float, default=10.0A (positive) parameter that downweights early iterations in onlinelearning. It should be greater than 1.0. In the literature, this iscalled tau_0.",10.0
,"max_iter max_iter: int, default=10The maximum number of passes over the training data (aka epochs).It only impacts the behavior in the :meth:`fit` method, and not the:meth:`partial_fit` method.",10
,"batch_size batch_size: int, default=128Number of documents to use in each EM iteration. Only used in onlinelearning.",128
,"evaluate_every evaluate_every: int, default=-1How often to evaluate perplexity. Only used in `fit` method.set it to 0 or negative number to not evaluate perplexity intraining at all. Evaluating perplexity can help you check convergencein training process, but it will also increase total training time.Evaluating perplexity in every iteration might increase training timeup to two-fold.",-1
,"total_samples total_samples: int, default=1e6Total number of documents. Only used in the :meth:`partial_fit` method.",1000000.0
,"perp_tol perp_tol: float, default=1e-1Perplexity tolerance. Only used when ``evaluate_every`` is greater than 0.",0.1


## Identifizierte Themen mit LDA

Die wichtigsten Begriffe pro Thema werden ausgegeben.

In [7]:
lda_words = bow_vectorizer.get_feature_names_out()

for index, topic in enumerate(lda.components_):
    print(f"\nTopic {index + 1}")

    print([
        lda_words[i]
        for i in topic.argsort()[-10:]
    ])


Topic 1
['open', 'number', 'report', 'inquiry', 'credit', 'balance', 'date', 'account', 'xxxxxxxx', 'xxxx']

Topic 2
['tell', 'would', 'bank', 'payment', 'pay', 'xxxxxxxx', 'loan', 'call', 'account', 'xxxx']

Topic 3
['right', 'agency', 'section', 'credit', 'usc', 'account', 'report', 'information', 'reporting', 'consumer']

Topic 4
['provide', 'error', 'financial', 'request', 'dispute', 'report', 'account', 'credit', 'late', 'payment']

Topic 5
['collection', 'dispute', 'file', 'remove', 'request', 'information', 'debt', 'account', 'credit', 'report']


## Themenanalyse mit Latent Semantic Analysis (LSA)

LSA analysiert semantische Zusammenhänge zwischen Begriffen und Dokumenten mithilfe linearer Algebra.

In [8]:
lsa = TruncatedSVD(
    n_components=5,
    random_state=42
)

In [9]:
lsa.fit(X_tfidf)

,"n_components n_components: int, default=2Desired dimensionality of output data.If algorithm='arpack', must be strictly less than the number of features.If algorithm='randomized', must be less than or equal to the number of features.The default value is useful for visualisation. For LSA, a value of100 is recommended.",5
,"algorithm algorithm: {'arpack', 'randomized'}, default='randomized'SVD solver to use. Either ""arpack"" for the ARPACK wrapper in SciPy(scipy.sparse.linalg.svds), or ""randomized"" for the randomizedalgorithm due to Halko (2009).",'randomized'
,"n_iter n_iter: int, default=5Number of iterations for randomized SVD solver. Not used by ARPACK. Thedefault is larger than the default in:func:`~sklearn.utils.extmath.randomized_svd` to handle sparsematrices that may have large slowly decaying spectrum.",5
,"n_oversamples n_oversamples: int, default=10Number of oversamples for randomized SVD solver. Not used by ARPACK.See :func:`~sklearn.utils.extmath.randomized_svd` for a completedescription... versionadded:: 1.1",10
,"power_iteration_normalizer power_iteration_normalizer: {'auto', 'QR', 'LU', 'none'}, default='auto'Power iteration normalizer for randomized SVD solver.Not used by ARPACK. See :func:`~sklearn.utils.extmath.randomized_svd`for more details... versionadded:: 1.1",'auto'
,"random_state random_state: int, RandomState instance or None, default=NoneUsed during randomized svd. Pass an int for reproducible results acrossmultiple function calls.See :term:`Glossary `.",42
,"tol tol: float, default=0.0Tolerance for ARPACK. 0 means machine precision. Ignored by randomizedSVD solver.",0.0


## Identifizierte Themen mit LSA

Die wichtigsten Begriffe der semantischen Komponenten werden ausgegeben.

In [10]:
lsa_words = tfidf_vectorizer.get_feature_names_out()

for index, component in enumerate(lsa.components_):
    print(f"\nTopic {index + 1}")

    print([
        lsa_words[i]
        for i in component.argsort()[-10:]
    ])


Topic 1
['payment', 'usc', 'reporting', 'consumer', 'information', 'xxxxxxxx', 'report', 'credit', 'account', 'xxxx']

Topic 2
['supply', 'tx', 'car', 'silence', 'estoppel', 'fl', 'doctrine', 'xxxxxxxxxxxx', 'xxxxxxxx', 'xxxx']

Topic 3
['xxxx', 'furnish', 'privacy', 'state', 'agency', 'right', 'reporting', 'consumer', 'section', 'usc']

Topic 4
['tell', 'bank', 'loan', 'late', 'section', 'state', 'pay', 'usc', 'call', 'payment']

Topic 5
['creditor', 'history', 'update', 'ensure', 'banking', 'code', 'report', 'error', 'late', 'payment']


## Interpretation der Themenanalyse

Die Themenanalyse zeigt, dass sich die Beschwerden hauptsächlich um Kreditberichte, Kontoprobleme, Zahlungsverhalten sowie Streitfälle mit Finanzinstituten und Auskunfteien drehen. Besonders häufig treten Begriffe wie `credit`, `account`, `report`, `payment`, `consumer` und `information` auf.

Die LDA-Analyse erzeugte klar abgegrenzte Themencluster. Dabei konnten insbesondere Themenbereiche wie fehlerhafte Kreditberichte, Zahlungsprobleme, Inkassoverfahren, Streitfälle („disputes“) sowie rechtliche Beschwerden gegenüber Auskunfteien identifiziert werden. Die Themen wirken dabei relativ klar voneinander getrennt und gut interpretierbar.

Die LSA-Analyse zeigte ähnliche Schwerpunkte, berücksichtigte jedoch stärker semantische Zusammenhänge zwischen den Begriffen. Dadurch wurden zusätzlich spezifischere Begriffe wie `privacy`, `banking`, `doctrine` oder `creditor` hervorgehoben. LSA eignet sich daher besonders zur Erkennung semantischer Beziehungen innerhalb großer Textmengen.

Insgesamt lieferten beide Verfahren ähnliche Kernthemen, wobei LDA besser für klar getrennte Themencluster geeignet war, während LSA tiefere semantische Zusammenhänge zwischen den Beschwerden sichtbar machte.

## Speicherung der Themenmodelle

Die erzeugten Themenmodelle werden gespeichert, damit sie für spätere Analysen und Visualisierungen erneut verwendet werden können.

In [11]:
with open("../processed_data/04_lda_model.pkl", "wb") as f:
    pickle.dump(lda, f)

In [12]:
with open("../processed_data/04_lsa_model.pkl", "wb") as f:
    pickle.dump(lsa, f)

In [13]:
with open("../results/lda_topics.txt", "w") as f:

    for index, topic in enumerate(lda.components_):

        words = [
            lda_words[i]
            for i in topic.argsort()[-10:]
        ]

        f.write(f"Topic {index + 1}: {words}\n\n")

In [14]:
with open("../results/lsa_topics.txt", "w") as f:

    for index, component in enumerate(lsa.components_):

        words = [
            lsa_words[i]
            for i in component.argsort()[-10:]
        ]

        f.write(f"Topic {index + 1}: {words}\n\n")